# Chapter 26: Deep Learning Capstone

Companion Colab notebook for **AI/ML: Zero to Hero**, Module 5 Capstone, Chapter 26.

This notebook actually *runs* the PyTorch code that the lesson page (`dl/ch26-deep-learning-capstone.html`) shows as reference-only. Every cell below was executed for real — no fabricated output.

This is the Module 5 capstone. You've built a neural network by hand with NumPy (Ch 21), automated it with PyTorch's tensors, autograd, and training loop (Ch 22), learned the architecture behind image models (Ch 23, CNNs) and sequence models (Ch 24, RNNs/LSTMs), and seen how attention replaces recurrence (Ch 25, Transformers). This chapter ties it all together: a complete, end-to-end image classifier — model, data loading, training loop, and evaluation.

**Note on data:** to keep this notebook fast, offline, and 100% reproducible, we use a small **synthetic** image dataset instead of downloading MNIST — but the pipeline shape (`DataLoader` → model → training loop → evaluation) is exactly what you'd use with a real dataset.

In [1]:
import torch
print('torch version:', torch.__version__)


torch version: 2.8.0+cpu


## 26.1 — What You've Built So Far

1. **Ch 21:** forward pass, loss, and backprop — by hand, with NumPy.
2. **Ch 22:** tensors, autograd, `nn.Module`, and the 5-line PyTorch training loop.
3. **Ch 23:** CNNs — `Conv2d`, padding/stride, pooling, and channels, for image data.
4. **Ch 24:** RNNs/LSTMs — hidden state, gates, and sequence framing, for sequential data.
5. **Ch 25:** attention and Transformers — replacing recurrence with direct position-to-position lookups.

Every one of those chapters focused on one new idea in isolation. A real project needs to combine **all** of them into a single pipeline: load data in batches, define a model, train it for multiple epochs, and *measure* whether it actually works on data it hasn't seen. That's this chapter.

## 26.2 — Batching Data With DataLoader

Training on the whole dataset at once wastes memory and gives noisy, infrequent weight updates. PyTorch's `DataLoader` wraps a `Dataset` and automatically splits it into shuffled mini-batches, so your training loop simply iterates over it.

In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Any dataset that returns (input, label) pairs works -- here, a small synthetic one
X = torch.randn(200, 1, 28, 28)
y = torch.randint(0, 10, (200,))
train_dataset = TensorDataset(X, y)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
for images, labels in train_loader:
    print(images.shape, labels.shape)   # (64, 1, 28, 28) (64,) -- one shuffled batch at a time
    break


torch.Size([64, 1, 28, 28]) torch.Size([64])


## 26.3 — The Model: Reusing Chapter 23's CNN

This capstone reuses the exact `SimpleCNN` architecture from Chapter 23 — `(Conv2d → ReLU → MaxPool2d) × 2 → Flatten → Linear → Linear` — because the architecture isn't the new idea here; putting it into a complete, working pipeline is.

In [3]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

print(SimpleCNN(num_classes=10))


SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


## 26.4 — The Full Training Loop, Over Batches and Epochs

Chapter 22's 5-line loop trained on one fixed tensor. A real training loop nests that same 5 lines inside a loop over `DataLoader` batches, which is itself inside a loop over epochs.

In [4]:
import torch.nn as nn
import torch.optim as optim

model = SimpleCNN(num_classes=10)
criterion = nn.CrossEntropyLoss()                       # standard loss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(criterion)
print(optimizer)


CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


> **model.train() vs. model.eval()** — some layers (like Dropout or BatchNorm) behave differently during training vs. inference. Calling `model.train()` before training and `model.eval()` before evaluation ensures consistent, correct behavior either way.

## 26.5 — Evaluation: Measuring Accuracy on Held-Out Data

A low training loss only tells you the model fit the data it trained on. To know if it actually *learned* something general, evaluate it on a separate test set it never trained on, with gradient tracking turned off (`torch.no_grad()`).

> **Never evaluate on the same data you trained on.** A model can memorize its training set and score perfectly there while failing completely on new examples — this is **overfitting**, and a held-out test set is the only way to catch it.

## 26.6 — Putting It All Together

The complete pipeline, in order, is: **DataLoader → model → criterion + optimizer → (epoch loop → batch loop → zero_grad/forward/loss/backward/step) → model.eval() + torch.no_grad() → accuracy**. This exact shape — with a bigger model, a bigger dataset, and more epochs — is what trains every image classifier, language model, and recommendation system you'll encounter after this course.

> **What would change for a real dataset?** Swap the synthetic `TensorDataset` for `torchvision.datasets.MNIST(...)` (or any real dataset), keep everything else identical. The pipeline shape doesn't change — only the data source does.

## Chapter 26 Summary — Module 5 Complete

1. **DataLoader** batches and shuffles a dataset so training doesn't need the whole dataset in memory at once.
2. The training loop nests the 5-line **zero_grad → forward → loss → backward → step** body inside a batch loop, inside an epoch loop.
3. **model.train()** and **model.eval()** switch the model between training and inference behavior.
4. **torch.no_grad()** disables gradient tracking during evaluation, since no backward pass is needed.
5. Always evaluate accuracy on a **held-out test set**, never on training data, to catch overfitting.

---

## Module 5 Capstone: End-to-End Image Classifier

The lesson's capstone trains on the real MNIST dataset in Colab. To keep this notebook **fast, offline, and 100% reproducible** (no network-dependent dataset download), we build the exact same pipeline — `DataLoader`, `SimpleCNN`, training loop, evaluation — on a fully synthetic, multi-class (harder than binary) image task generated entirely offline.

**Synthetic task:** each 28×28 "image" is random noise with one of its four quadrants (top-left / top-right / bottom-left / bottom-right) brightened — 4-class classification, requiring the CNN to localize *where* the bright region is, not just detect brightness.

**What we build and actually run:**
- `SimpleCNN(num_classes=4)` — the exact architecture from 26.3
- `DataLoader`-wrapped train and test sets (26.2)
- `criterion = nn.CrossEntropyLoss()`, `optimizer = optim.Adam(model.parameters(), lr=0.001)`
- The full epoch-over-batches training loop from 26.4
- `model.eval()` + `torch.no_grad()` evaluation on the held-out test set from 26.5, printing final test accuracy

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

NUM_CLASSES = 4

def make_dataset(n):
    """Synthetic 4-class 'which quadrant is brightest' task."""
    labels = torch.randint(0, NUM_CLASSES, (n,))
    imgs = torch.randn(n, 1, 28, 28) * 0.9
    for i in range(n):
        c = labels[i].item()
        if c == 0:
            imgs[i, 0, :14, :14] += 1.4   # top-left
        elif c == 1:
            imgs[i, 0, :14, 14:] += 1.4   # top-right
        elif c == 2:
            imgs[i, 0, 14:, :14] += 1.4   # bottom-left
        else:
            imgs[i, 0, 14:, 14:] += 1.4   # bottom-right
    return imgs, labels

X_train, y_train = make_dataset(600)
X_test, y_test = make_dataset(150)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

model = SimpleCNN(num_classes=NUM_CLASSES)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()                        # puts the model in "training mode"
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()            # 1. clear old gradients
        outputs = model(images)          # 2. forward pass
        loss = criterion(outputs, labels)   # 3. compute loss
        loss.backward()                  # 4. backward pass
        optimizer.step()                 # 5. update weights
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, avg loss {running_loss/len(train_loader):.4f}')

model.eval()                     # puts the model in "inference mode"
correct = 0
total = 0
with torch.no_grad():             # no gradients needed -- we're not calling .backward()
    for images, labels in test_loader:
        outputs = model(images)
        predicted = torch.argmax(outputs, dim=1)   # pick the highest-scoring class per example
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f'Test accuracy: {100 * correct / total:.2f}%')


Epoch 1, avg loss 0.3802


Epoch 2, avg loss 0.0000
Epoch 3, avg loss 0.0000


Epoch 4, avg loss 0.0000


Epoch 5, avg loss 0.0000
Test accuracy: 100.00%


Training loss drops sharply across epochs and the model reaches a high held-out test accuracy on 4-way classification (random guessing would be 25%) — confirming the complete pipeline (batched data loading, training over epochs, and evaluation on unseen data) genuinely works end to end.

## Module 5 complete. Next: Module 6 — NLP Foundations.